# Fine-tune Qwen2.5-7B-Instruct (LoRA) trên Kaggle TPU

**Settings:** Accelerator = *TPU VM* (v5e-8 hoặc v3-8), Internet = *On*, Add Data = dataset chứa thư mục `traindata`.

Chạy lần lượt từng cell (hoặc *Run All*). Kết quả nằm ở `/kaggle/working/output` (adapter tốt nhất, biểu đồ, `report.md`, `metrics.json`).

Nếu phiên Kaggle hết giờ giữa chừng: *Save Version* với output, chạy lại notebook — `RESUME=True` sẽ train tiếp từ checkpoint (cần đưa `/kaggle/working/output` của phiên trước vào lại, hoặc dùng *Persistence* của Kaggle).

---

Fine-tune Qwen2.5-7B-Instruct (LoRA) trên Kaggle TPU (PyTorch/XLA SPMD) theo idea.docx.


Khác với idea.docx (bắt buộc vì phần cứng):
- TPU không chạy được bitsandbytes 4-bit nên dùng LoRA trên trọng số bf16 (cùng r/alpha/dropout/target),
trọng số chia (shard) trên 8 lõi TPU bằng SPMD (tương đương FSDP).
Giữ đúng idea.docx: loss chỉ trên lượt assistant cuối (+ <|im_end|>), max 4096 token, cắt lịch sử cũ nhất khi quá dài,
LR 2e-4 cosine warmup 3%, batch hiệu dụng 16, 3 epoch + early stopping (patience 3) theo val loss, seed 42,
validation 10% mỗi nhóm, val loss riêng từng nhóm, test không dùng khi train.

## 0. Cài đặt thư viện

In [1]:
%pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 rouge-score==0.1.2 bert-score==0.3.13 nltk matplotlib
# Sau khi cài xong, nếu Kaggle báo cần restart kernel thì Restart rồi chạy tiếp từ cell dưới.


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Cấu hình

In [2]:
import os, glob, json, math, random, time, re, copy, functools
from collections import defaultdict, Counter

CFG = dict(
    MODEL_ID="Qwen/Qwen2.5-7B-Instruct",
    DATA_DIR=None,                     # None = tự tìm thư mục chứa group_A_formatted.jsonl trong /kaggle/input
    OUT_DIR="/kaggle/working/output",
    SEED=42,
    MAX_LEN=4096,
    # LoRA (idea.docx)
    LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.05,
    LORA_TARGETS=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # Huấn luyện
    NUM_EPOCHS=3,
    LR=2e-4, WARMUP_RATIO=0.03, WEIGHT_DECAY=0.0, MAX_GRAD_NORM=1.0,
    PER_DEVICE_BS=1,                   # batch mỗi lõi TPU; batch toàn cục = số lõi x PER_DEVICE_BS
    GRAD_ACCUM=2,                      # 8 lõi x 1 x 2 = batch hiệu dụng 16 (idea.docx: 16-32)
    VAL_RATIO=0.10,
    EVALS_PER_EPOCH=4,                 # đánh giá val 4 lần mỗi epoch
    EARLY_STOP_PATIENCE=3,             # dừng nếu val loss không giảm 3 lần đánh giá liên tiếp
    LOG_EVERY=10,                      # ghi train loss mỗi 10 bước optimizer
    # idea.docx đề xuất nhân B x3 khi B ~864 mẫu; hiện B (~3.770) đã ngang A (~3.450) nên để 1.
    OVERSAMPLE={"A": 1, "B1": 1, "B2": 1, "B3": 1, "B4": 1},
    NUM_BUCKETS=3,                     # số độ dài padding cố định (mỗi độ dài = 1 lần biên dịch XLA)
    TIME_LIMIT_HOURS=7.0,              # Kaggle giới hạn 9h/phiên: dừng train an toàn, chừa ~2h cho đánh giá
    RESUME=True,                       # nếu có checkpoint trong OUT_DIR thì train tiếp
    # Đánh giá
    EVAL_BASE_MODEL=True,              # so sánh với model gốc (perplexity + sinh văn bản)
    N_GEN_PER_GROUP=40,                # số mẫu test mỗi nhóm để sinh và chấm ROUGE/BERTScore (None = toàn bộ, rất lâu)
    GEN_MAX_NEW_TOKENS=1024,
    BERTSCORE_MODEL="xlm-roberta-large",  # đa ngôn ngữ, có tiếng Việt; chạy trên CPU
)

GROUP_FILES = {
    "A":  ("group_A_formatted.jsonl", "test_A_formatted.jsonl"),
    "B1": ("group_B_B1.jsonl", "test_B_B1.jsonl"),
    "B2": ("group_B_B2.jsonl", "test_B_B2.jsonl"),
    "B3": ("group_B_B3.jsonl", "test_B_B3.jsonl"),
    "B4": ("group_B_B4.jsonl", "test_B_B4.jsonl"),
}
GROUPS = list(GROUP_FILES)

FIXED_PROMPT = ("Bạn là chuyên gia tư vấn tâm lý. Luôn phản hồi bằng sự thấu cảm.\nCHỈ gợi ý thực phẩm khi ROUTE_NUTRITION = true, "
                "và CHỈ dùng thực phẩm có trong [NUTRITION_CONTEXT], kèm mã nguồn. Nếu có MISSING_STATES, khéo léo hỏi thêm 1–2 thông tin còn thiếu. Trả về đúng JSON.")
OUTPUT_KEYS = {"thought_process", "side_talk", "nutritional_advice", "source_reference"}

os.makedirs(CFG["OUT_DIR"], exist_ok=True)
REPORT_DIR = os.path.join(CFG["OUT_DIR"], "report"); os.makedirs(REPORT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CFG["OUT_DIR"], "checkpoint.pt")
BEST_DIR = os.path.join(CFG["OUT_DIR"], "best_adapter")
LOG_FILE = os.path.join(CFG["OUT_DIR"], "run_log.txt")


def log(*a):
    s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a)
    print(s, flush=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f: f.write(s + "\n")

## 2. Khởi tạo TPU (SPMD)

In [3]:
import numpy as np
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr
try:
    import torch_xla.distributed.spmd as xs            # torch_xla >= 2.3
except ImportError:
    import torch_xla.experimental.xla_sharding as xs   # torch_xla 2.1-2.2
from torch_xla.utils.checkpoint import checkpoint as xla_checkpoint

log("torch", torch.__version__, "| torch_xla", torch_xla.__version__)
xr.use_spmd()                                  # phải gọi trước khi tạo tensor trên TPU
DEVICE = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
N_DEV = xr.global_runtime_device_count()
MESH = xs.Mesh(np.arange(N_DEV), (N_DEV, 1), ("fsdp", "model"))
if hasattr(xs, "set_global_mesh"): xs.set_global_mesh(MESH)
GLOBAL_BS = N_DEV * CFG["PER_DEVICE_BS"]
log(f"TPU devices: {N_DEV} | global batch {GLOBAL_BS} | grad accum {CFG['GRAD_ACCUM']} | effective {GLOBAL_BS * CFG['GRAD_ACCUM']}")

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"]); xm.set_rng_state(CFG["SEED"])

/usr/local/lib/python3.12/site-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


09:37:26 torch 2.8.0+cpu | torch_xla 2.8.0


09:37:39 TPU devices: 8 | global batch 8 | grad accum 2 | effective 16


E0000 00:00:1791625051.104593      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


## 3. Dữ liệu, che loss và bucket độ dài

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup

if CFG["DATA_DIR"] is None:
    hits = glob.glob("/kaggle/input/**/group_A_formatted.jsonl", recursive=True)
    assert hits, "Không tìm thấy group_A_formatted.jsonl trong /kaggle/input — hãy Add Data thư mục traindata"
    CFG["DATA_DIR"] = os.path.dirname(hits[0])
log("DATA_DIR =", CFG["DATA_DIR"])


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


tok = AutoTokenizer.from_pretrained(CFG["MODEL_ID"])
IM_END_ID = tok.convert_tokens_to_ids("<|im_end|>")
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else tok.convert_tokens_to_ids("<|endoftext|>")


def validate_record(r):
    """Kiểm tra format idea.docx trước khi train."""
    ms = r["messages"]
    assert set(r) == {"messages"}, "có khóa ngoài messages"
    assert ms[0]["role"] == "system" and ms[0]["content"].startswith(FIXED_PROMPT + "\n\n"), "system prompt sai"
    assert ms[-1]["role"] == "assistant" and isinstance(ms[-1]["content"], str)
    o = json.loads(ms[-1]["content"])
    assert set(o) == OUTPUT_KEYS


def encode(messages):
    """prefix = template(messages[:-1], add_generation_prompt=True); full = template(messages).
    Label chỉ là phần JSON lượt assistant cuối + <|im_end|>; mọi thứ khác = -100."""
    prefix = tok.apply_chat_template(messages[:-1], add_generation_prompt=True, tokenize=True)
    full = tok.apply_chat_template(messages, tokenize=True)
    assert full[:len(prefix)] == prefix, "prefix không khớp chat template"
    # bỏ các token sau <|im_end|> cuối (dấu xuống dòng của template)
    last_end = len(full) - 1 - full[::-1].index(IM_END_ID)
    full = full[:last_end + 1]
    labels = [-100] * len(prefix) + full[len(prefix):]
    return full, labels


DROPPED = []


def build_example(r, group, src):
    msgs = copy.deepcopy(r["messages"])
    ids, labels = encode(msgs)
    # Quá 4096 token: bỏ cặp user/assistant lịch sử cũ nhất, giữ system + user cuối + assistant cuối
    while len(ids) > CFG["MAX_LEN"] and len(msgs) > 3:
        msgs = [msgs[0]] + msgs[3:]
        ids, labels = encode(msgs)
    if len(ids) > CFG["MAX_LEN"]:
        DROPPED.append({"src": src, "len": len(ids)}); return None
    return {"input_ids": ids, "labels": labels, "group": group, "messages": msgs,
            "n_turns": sum(m["role"] == "user" for m in msgs)}


def convo_key(r):
    return tuple(m["content"].strip() for m in r["messages"] if m["role"] == "user")


train_ex, val_ex, test_ex = [], defaultdict(list), defaultdict(list)
rng = random.Random(CFG["SEED"])
for g, (ftrain, ftest) in GROUP_FILES.items():
    rows = read_jsonl(os.path.join(CFG["DATA_DIR"], ftrain))
    trows = read_jsonl(os.path.join(CFG["DATA_DIR"], ftest))
    for r in rows + trows: validate_record(r)
    # tách validation theo hội thoại (các biến thể cùng chuỗi tin nhắn user nằm cùng một phía)
    keys = sorted({convo_key(r) for r in rows}); rng.shuffle(keys)
    val_keys = set(keys[:max(1, round(len(keys) * CFG["VAL_RATIO"]))])
    for i, r in enumerate(rows):
        ex = build_example(r, g, f"{ftrain}:{i+1}")
        if ex is None: continue
        if convo_key(r) in val_keys: val_ex[g].append(ex)
        else: train_ex.extend([ex] * CFG["OVERSAMPLE"][g])
    for i, r in enumerate(trows):
        ex = build_example(r, g, f"{ftest}:{i+1}")
        if ex is not None: test_ex[g].append(ex)
rng.shuffle(train_ex)
log("Train:", dict(Counter(e["group"] for e in train_ex)), "| total", len(train_ex))
log("Val:", {g: len(v) for g, v in val_ex.items()}, "| Test:", {g: len(v) for g, v in test_ex.items()})
log("Bị loại vì > MAX_LEN:", len(DROPPED), DROPPED[:5])

# ---- Kiểm tra che loss (idea.docx: bắt buộc trước khi train thật) ----
def show_masking(ex, title):
    lab = [t for t in ex["labels"] if t != -100]
    log(f"--- {title}: {len(ex['input_ids'])} token, {len(lab)} token tính loss ---")
    log("VĂN BẢN ĐÃ ÁP TEMPLATE (cuối):", tok.decode(ex["input_ids"])[-400:].replace("\n", "\\n"))
    log("PHẦN TÍNH LOSS:", tok.decode(lab)[:300].replace("\n", "\\n"), "...", tok.decode(lab[-5:]))

single = next(e for e in train_ex if e["n_turns"] == 1)
multi = next(e for e in train_ex if e["n_turns"] > 1)
show_masking(single, "Mẫu một lượt"); show_masking(multi, "Mẫu nhiều lượt")
prev_assist = [m["content"] for m in multi["messages"][1:-1] if m["role"] == "assistant"]
lab_text = tok.decode([t for t in multi["labels"] if t != -100])
assert not any(p[:50] in lab_text for p in prev_assist), "phần tính loss chứa lượt assistant cũ!"
for e in train_ex + [e for v in val_ex.values() for e in v] + [e for v in test_ex.values() for e in v]:
    lab = [t for t in e["labels"] if t != -100]
    assert lab and lab[-1] == IM_END_ID, "mẫu không có token tính loss"
    json.loads(tok.decode(lab[:-1]))  # phần tính loss parse được JSON
log("OK: mọi mẫu có token tính loss và phần đó là JSON hợp lệ + <|im_end|>")

# ---- Bucket độ dài (TPU cần shape cố định; mỗi bucket biên dịch 1 lần) ----
all_lens = sorted(len(e["input_ids"]) for e in train_ex)
qs = [all_lens[min(len(all_lens) - 1, int(len(all_lens) * q))] for q in np.linspace(0, 1, CFG["NUM_BUCKETS"] + 1)[1:]]
BUCKETS = sorted({min(CFG["MAX_LEN"], int(math.ceil(q / 128) * 128)) for q in qs})
if max(len(e["input_ids"]) for v in list(val_ex.values()) + list(test_ex.values()) for e in v) > BUCKETS[-1]:
    BUCKETS.append(CFG["MAX_LEN"])
log("Độ dài token: p50", all_lens[len(all_lens) // 2], "max", all_lens[-1], "| buckets", BUCKETS)


def bucket_of(n): return next(b for b in BUCKETS if b >= n)


def make_batches(examples, shuffle, seed):
    by_b = defaultdict(list)
    for e in examples: by_b[bucket_of(len(e["input_ids"]))].append(e)
    batches = []
    r = random.Random(seed)
    for b, xs_ in by_b.items():
        if shuffle: r.shuffle(xs_)
        for i in range(0, len(xs_), GLOBAL_BS):
            batches.append((b, xs_[i:i + GLOBAL_BS]))
    if shuffle: r.shuffle(batches)
    return batches


def collate(bucket, exs):
    """Pad phải tới độ dài bucket; thêm hàng giả (labels toàn -100) cho đủ batch toàn cục."""
    ids = torch.full((GLOBAL_BS, bucket), PAD_ID, dtype=torch.long)
    lab = torch.full((GLOBAL_BS, bucket), -100, dtype=torch.long)
    att = torch.zeros((GLOBAL_BS, bucket), dtype=torch.long)
    att[:, 0] = 1
    for i, e in enumerate(exs):
        n = len(e["input_ids"])
        ids[i, :n] = torch.tensor(e["input_ids"]); lab[i, :n] = torch.tensor(e["labels"]); att[i, :n] = 1
    ntok = int((lab[:, 1:] != -100).sum())
    return ids, att, lab, ntok


def to_device(ids, att, lab):
    ids, att, lab = ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)
    for t in (ids, att, lab): xs.mark_sharding(t, MESH, ("fsdp", None))
    return ids, att, lab

09:37:40 DATA_DIR = /kaggle/input/datasets/ghoenix/modelh


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

09:38:39 --- Mẫu nhiều lượt: 504 token, 214 token tính loss ---


09:38:39 VĂN BẢN ĐÃ ÁP TEMPLATE (cuối): a nhà và tình trạng chán ăn kéo dài. Câu trả lời xoa dịu sự kiệt sức, sau đó hỏi thăm về giấc ngủ."}, "side_talk": "Sự thất vọng hết lần này đến lần khác làm dạ dày bạn xót xa và tước đi sự ngon miệng, hành trình an cư lạc nghiệp ở thời buổi này thật sự quá gian nan. Mình muốn hiểu thêm về bạn một chút, đêm đến bạn thường ngủ thế nào?", "nutritional_advice": null, "source_reference": []}<|im_end|>


09:38:39 PHẦN TÍNH LOSS: {"thought_process": {"detected_emotions": "Nản lòng, chới với", "detected_symptoms": ["xót cồn cào dạ dày", "đắng chát miệng"], "trigger_nutrition": false, "reasoning": "ROUTE_NUTRITION = false. Người dùng nản lòng vì liên tục bị tranh mua nhà dẫn đến bỏ bữa, xót dạ dày và đắng miệng. Đã xác định đư ... _reference": []}<|im_end|>


09:38:51 OK: mọi mẫu có token tính loss và phần đó là JSON hợp lệ + <|im_end|>


09:38:51 Độ dài token: p50 477 max 844 | buckets [512, 896]


## 4. Model + LoRA

In [5]:
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

model = AutoModelForCausalLM.from_pretrained(CFG["MODEL_ID"], torch_dtype=torch.bfloat16,
                                             attn_implementation="eager", low_cpu_mem_usage=True)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=CFG["LORA_R"], lora_alpha=CFG["LORA_ALPHA"], lora_dropout=CFG["LORA_DROPOUT"],
                                         target_modules=CFG["LORA_TARGETS"], bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

# Chuyển từng tham số lên TPU và shard trọng số gốc theo trục fsdp (tránh chép nguyên model 15GB lên mỗi lõi)
# (torch_xla không cho gán p.data CPU -> TPU, nên thay cả Parameter trong từng module)
n_sharded = 0
for mod in model.modules():
    for pname, p in list(mod._parameters.items()):
        if p is None or p.device.type == "xla":
            continue
        newp = torch.nn.Parameter(p.detach().to(DEVICE), requires_grad=p.requires_grad)
        if newp.ndim >= 2 and not newp.requires_grad and newp.shape[0] % N_DEV == 0:
            xs.mark_sharding(newp, MESH, ("fsdp",) + (None,) * (newp.ndim - 1)); n_sharded += 1
        mod._parameters[pname] = newp
        del p
    xm.mark_step()
model.to(DEVICE)  # buffers (rotary inv_freq)
xm.mark_step()
assert all(p.device.type == "xla" for p in model.parameters())
log(f"Đã chuyển model lên TPU, {n_sharded} ma trận trọng số được shard trên {N_DEV} lõi")

# Shard activation theo batch ở đầu ra mỗi decoder layer (giống HF FSDPv2)
def _shard_out(mod, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    xs.mark_sharding(h, MESH, ("fsdp", None, None))
    return out

DECODER_LAYERS = model.base_model.model.model.layers
for layer in DECODER_LAYERS: layer.register_forward_hook(_shard_out)

# Gradient checkpointing dùng checkpoint của torch_xla
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
for m in model.modules():
    if hasattr(m, "_gradient_checkpointing_func"):
        m._gradient_checkpointing_func = xla_checkpoint

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"], betas=(0.9, 0.999))
steps_per_epoch = math.ceil(len(make_batches(train_ex, False, 0)) / CFG["GRAD_ACCUM"])
total_steps = steps_per_epoch * CFG["NUM_EPOCHS"]
scheduler = get_cosine_schedule_with_warmup(optimizer, int(CFG["WARMUP_RATIO"] * total_steps), total_steps)
eval_every = max(1, steps_per_epoch // CFG["EVALS_PER_EPOCH"])
log(f"steps/epoch {steps_per_epoch} | total {total_steps} | eval mỗi {eval_every} bước")


def lora_state_cpu():
    return {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}


def save_adapter(state, out_dir):
    from safetensors.torch import save_file
    os.makedirs(out_dir, exist_ok=True)
    model.peft_config["default"].save_pretrained(out_dir)
    save_file({k: v.contiguous() for k, v in state.items()}, os.path.join(out_dir, "adapter_model.safetensors"))
    tok.save_pretrained(out_dir)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


/tmp/ipykernel_73/2567144283.py:22: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


09:41:49 Đã chuyển model lên TPU, 198 ma trận trọng số được shard trên 8 lõi


09:41:49 steps/epoch 429 | total 1287 | eval mỗi 107 bước


/tmp/ipykernel_73/2567144283.py:24: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


## 5. Hàm đánh giá loss / perplexity

In [6]:
@torch.no_grad()
def eval_loss(examples_by_group):
    """Loss trung bình theo token (chỉ lượt assistant cuối) cho từng nhóm và toàn bộ."""
    model.eval()
    res, tot_nll, tot_tok = {}, 0.0, 0
    for g, exs in examples_by_group.items():
        nll = torch.zeros((), device=DEVICE); ntok_all = 0
        for b, chunk in make_batches(exs, False, 0):
            ids, att, lab, ntok = collate(b, chunk)
            ids, att, lab = to_device(ids, att, lab)
            out = model(input_ids=ids, attention_mask=att, labels=lab)
            nll = nll + out.loss.float() * ntok
            ntok_all += ntok
            xm.mark_step()
        v = nll.item()
        res[g] = v / ntok_all
        tot_nll += v; tot_tok += ntok_all
    res["ALL"] = tot_nll / tot_tok
    model.train()
    return res

## 6. Huấn luyện (3 epoch, early stopping)

In [7]:
history = {"train": [], "val": [], "lr": []}
state = {"step": 0, "epoch": 0, "batch_in_epoch": 0, "best_val": float("inf"), "bad_evals": 0, "best_step": 0}

if CFG["RESUME"] and os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location="cpu")
    set_peft_model_state_dict(model, {k: v.to(DEVICE) for k, v in ck["lora"].items()})
    optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"])
    history, state = ck["history"], ck["state"]
    log(f"Resume từ bước {state['step']} (epoch {state['epoch']})")


def save_checkpoint():
    xm.mark_step()
    ck = {"lora": lora_state_cpu(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
          "history": history, "state": state}
    xm.save(ck, CKPT_PATH)


t_start = time.time()
stop_reason = None
model.train()
for p in trainable:  # tạo sẵn .grad để đồ thị XLA giống nhau ở mọi micro-step
    p.grad = torch.zeros_like(p)

for epoch in range(state["epoch"], CFG["NUM_EPOCHS"]):
    batches = make_batches(train_ex, True, CFG["SEED"] + epoch)
    running, running_n = torch.zeros((), device=DEVICE), 0
    for bi in range(state["batch_in_epoch"] if epoch == state["epoch"] else 0, len(batches)):
        b, chunk = batches[bi]
        ids, att, lab, _ = collate(b, chunk)
        ids, att, lab = to_device(ids, att, lab)
        loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
        (loss / CFG["GRAD_ACCUM"]).backward()
        running = running + loss.detach().float(); running_n += 1
        xm.mark_step()
        if (bi + 1) % CFG["GRAD_ACCUM"] != 0 and bi + 1 != len(batches):
            continue
        torch.nn.utils.clip_grad_norm_(trainable, CFG["MAX_GRAD_NORM"])
        optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=False)
        xm.mark_step()
        state["step"] += 1; state["batch_in_epoch"] = bi + 1; state["epoch"] = epoch

        if state["step"] % CFG["LOG_EVERY"] == 0:
            tl = running.item() / running_n
            history["train"].append((state["step"], tl)); history["lr"].append((state["step"], scheduler.get_last_lr()[0]))
            running, running_n = torch.zeros((), device=DEVICE), 0
            el = (time.time() - t_start) / 60
            log(f"epoch {epoch+1} step {state['step']}/{total_steps} | train loss {tl:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | {el:.1f} phút")

        if state["step"] % eval_every == 0 or state["step"] == total_steps:
            vl = eval_loss(val_ex)
            history["val"].append((state["step"], vl))
            log(f"[VAL] step {state['step']} | " + " | ".join(f"{k} {v:.4f} (ppl {math.exp(v):.2f})" for k, v in vl.items()))
            if vl["ALL"] < state["best_val"] - 1e-4:
                state["best_val"], state["bad_evals"], state["best_step"] = vl["ALL"], 0, state["step"]
                save_adapter(lora_state_cpu(), BEST_DIR)
                log(f"  -> best mới, đã lưu {BEST_DIR}")
            else:
                state["bad_evals"] += 1
                log(f"  -> không cải thiện ({state['bad_evals']}/{CFG['EARLY_STOP_PATIENCE']})")
            save_checkpoint()
            if state["bad_evals"] >= CFG["EARLY_STOP_PATIENCE"]:
                stop_reason = "early_stopping"; break
        if (time.time() - t_start) / 3600 > CFG["TIME_LIMIT_HOURS"]:
            stop_reason = "time_limit"; save_checkpoint(); break
    if stop_reason: break
    state["epoch"], state["batch_in_epoch"] = epoch + 1, 0

log(f"Kết thúc train: {stop_reason or 'đủ epoch'} | best val {state['best_val']:.4f} tại bước {state['best_step']}")
if stop_reason == "time_limit":
    log("Hết giờ phiên Kaggle: chạy lại notebook (RESUME=True) để train tiếp từ checkpoint.")

# Nạp lại adapter tốt nhất
if os.path.exists(os.path.join(BEST_DIR, "adapter_model.safetensors")):
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, {k: v.to(DEVICE) for k, v in load_file(os.path.join(BEST_DIR, "adapter_model.safetensors")).items()})
    xm.mark_step()
json.dump({"history": history, "state": state, "buckets": BUCKETS, "cfg": CFG},
          open(os.path.join(REPORT_DIR, "train_history.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)

/tmp/ipykernel_73/2392492821.py:35: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


/tmp/ipykernel_73/2392492821.py:40: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


09:48:45 epoch 1 step 10/1287 | train loss 2.1174 | lr 5.26e-05 | 6.9 phút


09:50:39 epoch 1 step 20/1287 | train loss 1.5113 | lr 1.05e-04 | 8.8 phút


09:50:58 epoch 1 step 30/1287 | train loss 1.2053 | lr 1.58e-04 | 9.1 phút


...

10:37:31 epoch 3 step 1170/1287 | train loss 0.4695 | lr 4.30e-06 | 55.7 phút


10:38:04 [VAL] step 1177 | A 0.8766 (ppl 2.40) | B1 0.7411 (ppl 2.10) | B2 0.0178 (ppl 1.02) | B3 0.5515 (ppl 1.74) | B4 0.5991 (ppl 1.82) | ALL 0.6873 (ppl 1.99)


10:38:04   -> không cải thiện (3/3)


10:38:15 Kết thúc train: early_stopping | best val 0.6687 tại bước 856


/tmp/ipykernel_73/2392492821.py:77: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


## 7. Perplexity trên tập test

In [8]:
metrics = {"test_loss_ft": eval_loss(test_ex)}
metrics["test_ppl_ft"] = {g: math.exp(v) for g, v in metrics["test_loss_ft"].items()}
if CFG["EVAL_BASE_MODEL"]:
    with model.disable_adapter():
        metrics["test_loss_base"] = eval_loss(test_ex)
    metrics["test_ppl_base"] = {g: math.exp(v) for g, v in metrics["test_loss_base"].items()}
log("Perplexity test (fine-tune):", {k: round(v, 3) for k, v in metrics["test_ppl_ft"].items()})
if CFG["EVAL_BASE_MODEL"]: log("Perplexity test (gốc):", {k: round(v, 3) for k, v in metrics["test_ppl_base"].items()})

/tmp/ipykernel_73/2597872135.py:14: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


10:42:31 Perplexity test (fine-tune): {'A': 2.342, 'B1': 2.632, 'B2': 3.221, 'B3': 1.937, 'B4': 1.957, 'ALL': 2.412}


10:42:31 Perplexity test (gốc): {'A': 9.938, 'B1': 14.229, 'B2': 9.513, 'B3': 13.377, 'B4': 19.763, 'ALL': 11.229}


## 8. Sinh văn bản trên tập test

In [9]:
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()
tok.padding_side = "left"

gen_sets = {}
for g in GROUPS:
    exs = test_ex[g]
    n = CFG["N_GEN_PER_GROUP"] or len(exs)
    gen_sets[g] = random.Random(CFG["SEED"]).sample(exs, min(n, len(exs)))


@torch.no_grad()
def generate_texts(exs, use_adapter=True):
    outs = []
    prompts = [tok.apply_chat_template(e["messages"][:-1], add_generation_prompt=True, tokenize=False) for e in exs]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    res = [None] * len(prompts)
    for s in range(0, len(order), GLOBAL_BS):
        idx = order[s:s + GLOBAL_BS]
        batch = [prompts[i] for i in idx] + [prompts[idx[-1]]] * (GLOBAL_BS - len(idx))
        plen = max(len(tok(p, add_special_tokens=False)["input_ids"]) for p in batch)
        enc = tok(batch, add_special_tokens=False, padding="max_length", max_length=bucket_of(plen), return_tensors="pt")
        ids, att = enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)
        kw = dict(max_new_tokens=CFG["GEN_MAX_NEW_TOKENS"], do_sample=False, pad_token_id=PAD_ID,
                  eos_token_id=[IM_END_ID, tok.convert_tokens_to_ids("<|endoftext|>")])
        def _run():
            try:
                return model.generate(input_ids=ids, attention_mask=att, cache_implementation="static", **kw)
            except Exception as ex:  # fallback: cache động (chậm hơn nhiều trên TPU)
                log("static cache lỗi, dùng cache động:", repr(ex)[:200])
                return model.generate(input_ids=ids, attention_mask=att, **kw)
        if use_adapter: out = _run()
        else:
            with model.disable_adapter(): out = _run()
        out = out[:, ids.shape[1]:].cpu()
        for j, i in enumerate(idx):
            res[i] = tok.decode(out[j], skip_special_tokens=True).strip()
        log(f"  sinh {min(s + GLOBAL_BS, len(order))}/{len(order)}")
    return res


generations = {}
for g in GROUPS:
    log(f"Sinh văn bản nhóm {g} (fine-tune) ...")
    generations[("ft", g)] = generate_texts(gen_sets[g], True)
    if CFG["EVAL_BASE_MODEL"]:
        log(f"Sinh văn bản nhóm {g} (gốc) ...")
        generations[("base", g)] = generate_texts(gen_sets[g], False)

10:42:31 Sinh văn bản nhóm A (fine-tune) ...


/usr/local/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


## 9. ROUGE, BERTScore và kiểm tra luật

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn


class ViTokenizer:
    """Tách từ unicode (tokenizer mặc định của rouge-score xóa dấu tiếng Việt)."""
    def tokenize(self, text):
        return re.findall(r"\w+", text.lower(), flags=re.UNICODE)


SCORER = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ViTokenizer())


def parse_output(text):
    t = text.strip()
    if t.startswith("```"): t = re.sub(r"^```(json)?|```$", "", t).strip()
    try:
        o = json.loads(t)
        if isinstance(o, dict): return o
    except Exception:
        pass
    return None


def response_text(o, raw):
    """Văn bản dùng để so khớp: side_talk (+ nutritional_advice nếu có); không parse được thì dùng nguyên văn."""
    if o is None: return raw
    parts = [str(o.get("side_talk") or "")]
    if o.get("nutritional_advice"): parts.append(str(o["nutritional_advice"]))
    return " ".join(parts).strip()


FOOD_RE = r"\b(ăn|uống|cháo|trà|sữa|cà phê|thực phẩm|món|thức uống|đồ uống|dưỡng chất|vitamin|rau|trái cây)\b"


def rule_metrics(e, o):
    sysc = e["messages"][0]["content"]
    route = "[ROUTE_NUTRITION]: true" in sysc
    m = {"json_valid": o is not None and set(o) == OUTPUT_KEYS and isinstance(o.get("thought_process"), dict)}
    if not m["json_valid"]:
        return m | {"trigger_match": False, "advice_rule": False}
    tp = o["thought_process"]
    m["trigger_match"] = tp.get("trigger_nutrition") is route
    if route:
        codes = set(re.findall(r"(NUT_\w+|PSY_\w+)", sysc))
        refs = o.get("source_reference") or []
        m["advice_rule"] = bool(o.get("nutritional_advice")) and bool(refs) and all(c in codes for c in refs) \
            and not re.search(r"\d\s*(mg|g|kcal)\b", str(o.get("nutritional_advice")))
    else:
        m["advice_rule"] = o.get("nutritional_advice") is None and o.get("source_reference") == []
    if e["group"] == "B3":
        m["b3_no_food"] = not re.search(FOOD_RE, str(o.get("side_talk", "")), re.I)
    return m


gen_rows, gen_metrics = [], {}
for kind in (["ft", "base"] if CFG["EVAL_BASE_MODEL"] else ["ft"]):
    all_c, all_r, owner = [], [], []
    for g in GROUPS:
        for e, raw in zip(gen_sets[g], generations[(kind, g)]):
            ref = json.loads(e["messages"][-1]["content"])
            o = parse_output(raw)
            c, r = response_text(o, raw), response_text(ref, "")
            rs = SCORER.score(r, c)
            row = {"model": kind, "group": g, "prediction": raw, "reference": e["messages"][-1]["content"],
                   "rouge1": rs["rouge1"].fmeasure, "rouge2": rs["rouge2"].fmeasure, "rougeL": rs["rougeL"].fmeasure}
            row.update(rule_metrics(e, o))
            gen_rows.append(row); all_c.append(c or " "); all_r.append(r or " "); owner.append(len(gen_rows) - 1)
    log(f"BERTScore ({kind}) trên CPU với {CFG['BERTSCORE_MODEL']} ...")
    P, R, F = bert_score_fn(all_c, all_r, model_type=CFG["BERTSCORE_MODEL"], device="cpu", batch_size=16, verbose=False)
    for k, i in enumerate(owner):
        gen_rows[i]["bertscore_f1"] = float(F[k])
    for g in GROUPS + ["ALL"]:
        rows = [r for r in gen_rows if r["model"] == kind and (g == "ALL" or r["group"] == g)]
        agg = {}
        for key in ["rouge1", "rouge2", "rougeL", "bertscore_f1", "json_valid", "trigger_match", "advice_rule", "b3_no_food"]:
            vals = [float(r[key]) for r in rows if key in r]
            if vals: agg[key] = sum(vals) / len(vals)
        agg["n"] = len(rows)
        gen_metrics[f"{kind}/{g}"] = agg

metrics["generation"] = gen_metrics
with open(os.path.join(REPORT_DIR, "generations.jsonl"), "w", encoding="utf-8") as f:
    for r in gen_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
json.dump(metrics, open(os.path.join(REPORT_DIR, "metrics.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
log("Generation metrics:", json.dumps(gen_metrics, ensure_ascii=False))

## 10. Biểu đồ và báo cáo

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

C_FT, C_BASE = "#2563eb", "#9ca3af"
GCOL = {"A": "#2563eb", "B1": "#16a34a", "B2": "#ea580c", "B3": "#dc2626", "B4": "#7c3aed", "ALL": "#111827"}


def savefig(fig, name):
    p = os.path.join(REPORT_DIR, name); fig.tight_layout(); fig.savefig(p, dpi=150); plt.close(fig); return p


def smooth(y, k=5):
    if len(y) < k: return y
    return list(np.convolve(y, np.ones(k) / k, mode="valid"))


figs = []
# (1) train loss + LR
if history["train"]:
    s, l = zip(*history["train"])
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(s, l, color=C_FT, alpha=.35, label="train loss")
    sm = smooth(list(l)); ax.plot(s[len(s) - len(sm):], sm, color=C_FT, lw=2, label="train loss (trung bình trượt)")
    if history["val"]:
        vs = [x[0] for x in history["val"]]; vv = [x[1]["ALL"] for x in history["val"]]
        ax.plot(vs, vv, "o-", color="#ea580c", lw=2, label="val loss")
        ax.axvline(state["best_step"], color="#16a34a", ls="--", label=f"best (bước {state['best_step']})")
    for e in range(1, CFG["NUM_EPOCHS"]):
        ax.axvline(e * steps_per_epoch, color="#d1d5db", lw=1)
    ax.set_xlabel("bước optimizer"); ax.set_ylabel("loss (lượt assistant cuối)"); ax.set_title("Train / validation loss")
    ax2 = ax.twinx(); ls, lr = zip(*history["lr"]); ax2.plot(ls, lr, color="#6b7280", ls=":", label="learning rate"); ax2.set_ylabel("learning rate")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, fontsize=8)
    figs.append(savefig(fig, "1_train_val_loss.png"))
# (2) val loss + perplexity theo nhóm
if history["val"]:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    vs = [x[0] for x in history["val"]]
    for g in GROUPS + ["ALL"]:
        vv = [x[1][g] for x in history["val"]]
        axs[0].plot(vs, vv, "o-", color=GCOL[g], lw=2.5 if g == "ALL" else 1.5, label=g)
        axs[1].plot(vs, [math.exp(v) for v in vv], "o-", color=GCOL[g], lw=2.5 if g == "ALL" else 1.5, label=g)
    axs[0].set_title("Val loss theo nhóm"); axs[1].set_title("Val perplexity theo nhóm")
    for a in axs: a.set_xlabel("bước optimizer"); a.legend(fontsize=8)
    figs.append(savefig(fig, "2_val_by_group.png"))
# (3) perplexity test
fig, ax = plt.subplots(figsize=(9, 4.5))
xg = np.arange(len(GROUPS) + 1); w = 0.38; labels = GROUPS + ["ALL"]
ft = [metrics["test_ppl_ft"][g] for g in labels]
if CFG["EVAL_BASE_MODEL"]:
    bs = [metrics["test_ppl_base"][g] for g in labels]
    b1 = ax.bar(xg - w / 2, bs, w, color=C_BASE, label="Qwen2.5-7B gốc"); ax.bar_label(b1, fmt="%.2f", fontsize=8)
b2 = ax.bar(xg + (w / 2 if CFG["EVAL_BASE_MODEL"] else 0), ft, w, color=C_FT, label="Fine-tune"); ax.bar_label(b2, fmt="%.2f", fontsize=8)
ax.set_xticks(xg); ax.set_xticklabels(labels); ax.set_ylabel("perplexity (thấp hơn = tốt hơn)")
ax.set_title("Perplexity trên tập test (chỉ lượt assistant cuối)"); ax.set_yscale("log"); ax.legend()
figs.append(savefig(fig, "3_test_perplexity.png"))
# (4) ROUGE + BERTScore
keys = [("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeL", "ROUGE-L"), ("bertscore_f1", "BERTScore F1")]
fig, axs = plt.subplots(1, 4, figsize=(18, 4.5), sharey=False)
for a, (k, t) in zip(axs, keys):
    f_ = [gen_metrics[f"ft/{g}"].get(k, 0) for g in labels]
    if CFG["EVAL_BASE_MODEL"]:
        b_ = [gen_metrics[f"base/{g}"].get(k, 0) for g in labels]
        a.bar(xg - w / 2, b_, w, color=C_BASE, label="gốc")
    bb = a.bar(xg + (w / 2 if CFG["EVAL_BASE_MODEL"] else 0), f_, w, color=C_FT, label="fine-tune")
    a.bar_label(bb, fmt="%.2f", fontsize=7); a.set_xticks(xg); a.set_xticklabels(labels); a.set_title(t); a.set_ylim(0, 1)
axs[0].legend(fontsize=8)
fig.suptitle(f"Độ tương đồng với câu trả lời tham chiếu (side_talk + nutritional_advice), {CFG['N_GEN_PER_GROUP']} mẫu/nhóm")
figs.append(savefig(fig, "4_rouge_bertscore.png"))
# (5) luật idea.docx
rk = [("json_valid", "JSON hợp lệ"), ("trigger_match", "trigger khớp ROUTE"), ("advice_rule", "luật nutritional_advice")]
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
for a, (k, t) in zip(axs, rk):
    f_ = [gen_metrics[f"ft/{g}"].get(k, 0) for g in labels]
    if CFG["EVAL_BASE_MODEL"]:
        a.bar(xg - w / 2, [gen_metrics[f"base/{g}"].get(k, 0) for g in labels], w, color=C_BASE, label="gốc")
    bb = a.bar(xg + (w / 2 if CFG["EVAL_BASE_MODEL"] else 0), f_, w, color=C_FT, label="fine-tune")
    a.bar_label(bb, fmt="%.2f", fontsize=7); a.axhline(0.99, color="#dc2626", ls="--", lw=1)
    a.set_xticks(xg); a.set_xticklabels(labels); a.set_ylim(0, 1.05); a.set_title(t)
axs[0].legend(fontsize=8); fig.suptitle("Kiểm tra theo luật (ngưỡng 99% theo idea.docx)")
figs.append(savefig(fig, "5_rule_metrics.png"))

# report.md
L = ["# Báo cáo fine-tune Qwen2.5-7B-Instruct (LoRA, TPU)", "",
     f"- Dừng: **{stop_reason or 'đủ ' + str(CFG['NUM_EPOCHS']) + ' epoch'}**, best val loss {state['best_val']:.4f} tại bước {state['best_step']}/{total_steps}",
     f"- Train {len(train_ex)} mẫu, val {sum(len(v) for v in val_ex.values())}, test {sum(len(v) for v in test_ex.values())}; buckets {BUCKETS}",
     f"- LoRA r={CFG['LORA_R']} alpha={CFG['LORA_ALPHA']} dropout={CFG['LORA_DROPOUT']}; LR {CFG['LR']} cosine, warmup {CFG['WARMUP_RATIO']:.0%}; batch hiệu dụng {GLOBAL_BS * CFG['GRAD_ACCUM']}", "",
     "## Perplexity trên test", "", "| Nhóm | Gốc | Fine-tune |", "|---|---|---|"]
for g in labels:
    L.append(f"| {g} | {metrics.get('test_ppl_base', {}).get(g, float('nan')):.3f} | {metrics['test_ppl_ft'][g]:.3f} |")
L += ["", "## Sinh văn bản", "", "| Model | Nhóm | n | ROUGE-1 | ROUGE-2 | ROUGE-L | BERTScore F1 | JSON hợp lệ | trigger khớp | luật advice |", "|---|---|---|---|---|---|---|---|---|---|"]
for k, v in gen_metrics.items():
    kind, g = k.split("/")
    L.append(f"| {kind} | {g} | {v['n']} | {v.get('rouge1', 0):.3f} | {v.get('rouge2', 0):.3f} | {v.get('rougeL', 0):.3f} | {v.get('bertscore_f1', 0):.3f} | "
             f"{v.get('json_valid', 0):.1%} | {v.get('trigger_match', 0):.1%} | {v.get('advice_rule', 0):.1%} |")
L += ["", "## Biểu đồ", ""] + [f"![{os.path.basename(p)}]({os.path.basename(p)})" for p in figs]
open(os.path.join(REPORT_DIR, "report.md"), "w", encoding="utf-8").write("\n".join(L))
log("Xong. Báo cáo:", os.path.join(REPORT_DIR, "report.md"))

## Xem kết quả

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(os.path.join(REPORT_DIR, "report.md"), encoding="utf-8").read().split("## Biểu đồ")[0]))
for p in figs:
    display(Image(filename=p))